# Lecture: Comparing Distributions Across Groups

In the previous lecture, we compared two categorical features. Here we compare the distribution of one quantitative feature across categorical groups while following the same reproducible analysis workflow.


## Learning goals

- compare groups using center, spread, shape, overlap, and unusual values;
- create histograms, box plots, and violin plots with Seaborn;
- explain the strengths and limitations of each plot;
- interpret daily values that summarize many individual flights; and
- state a supported conclusion and limitation.


## Prepare the notebook

**Seaborn** is a visualization library built on Matplotlib. It works especially well with Pandas DataFrames and makes statistical comparisons among groups easier to create with concise code.

Import Pandas using its conventional alias `pd`, Matplotlib's `pyplot` component using its conventional alias `plt`, and Seaborn using its conventional alias `sns`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Airport taxi-in times in 2025

The Bureau of Transportation Statistics collects on-time performance for nonstop domestic flights reported by covered U.S. airlines. The teaching file summarizes those flight records by destination airport and date.

We will compare four major airports with visibly different taxi-in patterns:

- `MIA`: Miami International;
- `MCO`: Orlando International;
- `DFW`: Dallas–Fort Worth International; and
- `ORD`: Chicago O'Hare International.

Source: https://www.transtats.bts.gov/ONTIME/


## What is taxi-in time?

Taxi-in time is the number of minutes from **wheels-on**—when an arriving aircraft reaches the runway—to its arrival at the gate. It is the part of the trip after landing but before passengers can leave the aircraft.

The teaching file first calculates the mean taxi-in time among completed, non-diverted arrivals for each airport and date. Therefore, one value such as `12.4` means that arriving flights took an average of 12.4 minutes to travel from the runway to the gate at that airport on that date.


## The reproducible analysis workflow

1. **Question**
2. **Data**
3. **Operation**
4. **Check**
5. **Evidence**
6. **Conclusion**
7. **Limitation**


## Step 1 — Question

> **How did average daily taxi-in times differ among Miami, Orlando, Dallas–Fort Worth, and Chicago O'Hare in 2025?**

Landing does not mark the end of a passenger's trip. Airport layout, runway configuration, congestion, and gate availability can affect how long an aircraft spends traveling to the gate. Comparing complete distributions reveals typical differences as well as unusually slow days.


## Step 2 — Data


In [ ]:
airport_days = pd.read_csv(
    "data/bts_airport_daily_delays_2025.csv",
    parse_dates=["date"]
)
airport_days.head()


CSV files do not store a dedicated datetime data type, so Pandas would ordinarily read the dates as text. `parse_dates=["date"]` tells `read_csv()` to convert the `date` column to Pandas datetime values while loading the file. That conversion makes datetime tools such as `.dt.month` available later.


In [ ]:
airport_days.tail()


In [ ]:
airport_days.info()


In [ ]:
airport_days.isnull().sum()


## Data dictionary

The features used in this lesson are:

| Feature | Meaning |
| --- | --- |
| `airport` | Name of the destination airport |
| `date` | Calendar date |
| `month` | Month name |
| `day_of_week` | Day name |
| `mean_taxi_in_min` | Mean time from runway arrival to the gate |


## Discussion: check the data

Complete the table using `.info()`, `.isnull().sum()`, and the data dictionary. In the data type column, classify the feature as categorical, quantitative, or temporal. In the data storage type column, record the type reported by Pandas.

| Feature | Data type | Data storage type | Missing values |
| --- | --- | --- | ---: |
| `airport` |  |  |  |
| `date` |  |  |  |
| `month` |  |  |  |
| `day_of_week` |  |  |  |
| `mean_taxi_in_min` |  |  |  |


**What does each record represent: one flight, one airport, or something more specific? How can you tell?**


**Notes and calculations:**


### Discussion notes

| Feature | Data type | Data storage type | Missing values |
| --- | --- | --- | ---: |
| `airport` | Categorical | `str` or `object` | 0 |
| `date` | Temporal | `datetime64` | 0 |
| `month` | Categorical | `str` or `object` | 0 |
| `day_of_week` | Categorical | `str` or `object` | 0 |
| `mean_taxi_in_min` | Quantitative | `float64` | 0 |

Each record is an airport-day summary: one destination airport on one calendar date. It is not one flight. For example, the Orlando record for July 1 summarizes all qualifying flights scheduled to arrive at Orlando on July 1.


## Step 3 — Operation

Select the four airports. `.isin()` checks whether each airport name appears in our list.


In [ ]:
airport_order = [
    "Chicago O'Hare",
    "Dallas-Fort Worth",
    "Miami",
    "Orlando"
]

comparison = airport_days[
    airport_days["airport"].isin(airport_order)
].copy()

comparison["airport"].value_counts()


Filtering can produce a DataFrame that still refers back to the original object's data. We add `.copy()` when we create `comparison` so it is an independent working DataFrame that could be modified without changing `airport_days` or triggering an ambiguous chained-assignment warning. We do not actually modify `comparison` in this analysis, so `.copy()` is not strictly necessary here. It becomes especially useful when you plan to add, remove, or change values in a filtered subset.

The frequency table confirms that every airport contributes the same number of daily records. Seaborn must also assign categorical labels to plot positions. Text categories normally follow their first appearance in the data unless a Pandas categorical order or an `order` argument supplies another order. We pass `airport_order` explicitly so every figure remains consistent.


## Discussion: predict the longest taxi time

Which airport do you predict will have the longest typical taxi-in time? What airport characteristic informed your prediction?


### Discussion notes

Predictions may reasonably refer to airport size, runway layout, congestion, or prior travel experience. The prediction is a hypothesis to compare with the data, not evidence by itself.

Calculate the mean of `mean_taxi_in_min` for each airport. After `groupby()` separates the rows by airport, `.mean()` calculates one average for each group.

`.reindex(airport_order)` rearranges the result's existing rows to match `airport_order`. It does not recalculate the means or sort them by their values. If the list requested a label missing from the result, Pandas would add that label with a `NaN` value.


In [ ]:
taxi_summary = (
    comparison.groupby("airport")["mean_taxi_in_min"]
    .mean()
    .reindex(airport_order)
)
taxi_summary.round(1)


## Discussion: is the mean enough?

1. Does an airport's mean imply that taxi-in time is always that long?

2. What does the mean leave unanswered about how much taxi-in time fluctuates from day to day?

3. What additional features of each distribution should we compare?


### Discussion notes

1. No. A mean is one summary of all days, not the value observed every day.

2. The mean does not reveal the spread, range, skew, overlap, or unusually long days. Two airports could have similar means but very different day-to-day variability.

3. We should compare center, spread, shape, overlap, and unusual values using numerical summaries and figures.


## Step 4 — Check


In [ ]:
comparison["mean_taxi_in_min"].describe()


The earlier missing-value check found no missing values, and the frequency table confirmed that each airport contributes 365 dates. Because taxi-in time cannot be negative, the minimum and maximum from `.describe()` help us check that the observed range is possible and reasonable for daily averages.


## Step 5 — Evidence

### Histograms: detailed shape

The histogram divides taxi-in times into intervals called **bins**. The y-axis shows the number of days whose average taxi-in time falls within each bin. Because every airport contributes 365 days, the group counts are directly comparable.

The arguments used below control the plot:

- `x` selects the quantitative feature placed into bins;
- `hue` uses a different color for each airport and uses the feature name as the legend title;
- `hue_order` keeps the legend and drawing order consistent;
- `element="step"` draws each histogram with a stepped outline, making overlapping boundaries easier to distinguish;
- `multiple="layer"` places the airport histograms on the same axes so they can overlap;
- `alpha=0.45` makes the bars partly transparent—0 is completely transparent and 1 is completely opaque—so covered bars remain visible; and
- `palette="deep"` selects Seaborn's named set of colors.

Seaborn creates and labels a legend automatically when `hue` is used. `plt.gca().get_legend().set_title("Airport")` retrieves that existing legend and changes its title to the reader-friendly label `Airport`.


In [ ]:
sns.histplot(
    data=comparison,
    x="mean_taxi_in_min",
    hue="airport",
    hue_order=airport_order,
    element="step",
    multiple="layer",
    alpha=0.45,
    palette="deep"
)
plt.gca().get_legend().set_title("Airport")
plt.title("Distribution of Average Daily Taxi-In Time")
plt.xlabel("Average taxi-in time per completed arrival (minutes)")
plt.ylabel("Number of days")
plt.show()


## Discussion: interpret the histogram

1. Where does each airport's distribution appear to be centered?

2. Which distributions overlap substantially?

3. What information about shape is visible here that a single average would hide?


### Discussion notes

1. Miami and Orlando concentrate near roughly 9–10 minutes, Dallas–Fort Worth near 13 minutes, and Chicago O'Hare near 19 minutes.

2. Miami and Orlando overlap substantially. Dallas–Fort Worth overlaps their upper ranges, while Chicago O'Hare is more separated.

3. The histogram reveals spread, overlap, right skew, concentrations, and long upper tails that one average cannot show.


### Box plots: compact summaries

Statistician John Tukey developed the modern box plot as part of his approach to **exploratory data analysis**: using calculations and graphics to examine patterns before settling on an explanation or model. Tukey worked at Princeton and Bell Labs, contributed to the fast Fourier transform, and introduced the computing term *bit*. His work is an important bridge among statistics, data analysis, and computer science.

A box plot summarizes a distribution using several features:

- the line inside the box is the **median**, or 50th percentile;
- the two ends of the box are the **first quartile** (25th percentile) and **third quartile** (75th percentile);
- the length of the box is the **interquartile range**, or `IQR = Q3 - Q1`, containing the middle 50% of observations;
- the **whiskers** extend to the most extreme observed values within 1.5 IQRs of the box; and
- individual points beyond the whiskers are **potential outliers**.

“Potential outlier” is a reason to investigate a value, not proof that it is incorrect. The whiskers also do not automatically represent the minimum and maximum.


In [ ]:
sns.boxplot(
    data=comparison,
    x="mean_taxi_in_min",
    y="airport",
    order=airport_order,
    color="mediumseagreen"
)
plt.title("Average Daily Taxi-In Time by Airport")
plt.xlabel("Average taxi-in time per completed arrival (minutes)")
plt.ylabel("Destination airport")
plt.show()


## Discussion: histograms and box plots

1. Which distribution features are easier to see in the histograms?

2. Which distribution features are easier to see in the box plots?

3. What information does each figure make harder to see?


### Discussion notes

1. Histograms make modes, concentrations, detailed shape, skew, gaps, overlap, and relative group size easier to see. Their appearance can change with the bins, and overlapping groups can become busy.

2. Box plots make medians, quartiles, the middle 50%, overall compact comparisons, and potential outliers easier to see.

3. Histograms do not show exact medians or quartiles clearly. Box plots hide modes, detailed shape, gaps, and group size unless that information is added separately.


## Discussion: compare the box plots

1. Which airport has the greatest median taxi-in time?

2. Which airport has the widest middle 50% of days?

3. How do the potential outliers affect what you would say about a typical day?


### Discussion notes

1. Chicago O'Hare has the greatest median.

2. Chicago O'Hare has the widest middle 50% among the four airports.

3. Potential outliers show that unusually long days occurred, but they do not change the median into a description of those unusual days. A useful account should distinguish typical conditions from occasional extremes.


### Violin plots: an attempt to combine both views

A violin plot attempts to combine the strengths of a histogram and a box plot in one figure: its width represents a smoothed estimate of the distribution's shape, while its interior provides a compact box-plot summary.


In [ ]:
sns.violinplot(
    data=comparison,
    x="mean_taxi_in_min",
    y="airport",
    order=airport_order,
    color="mediumpurple"
)
plt.title("Average Daily Taxi-In Time by Airport")
plt.xlabel("Average taxi-in time per completed arrival (minutes)")
plt.ylabel("Destination airport")
plt.show()


With its default settings, Seaborn draws a small box-plot summary inside each violin and allows the smoothed curve to extend beyond the most extreme observed values. Those extensions come from the density estimate; they are not additional observations.

**Strengths:** provides a compact comparison of smoothed shape and a box-plot summary.

**Limitations:** the density is estimated, smoothing can hide rare disruption days, and many audiences are unfamiliar with violin widths.


## Discussion: choose among the figures

Which figure would you choose to communicate the airport comparison to a general audience? What does your choice show especially well, and what does it hide?


### Discussion notes

A box plot is a reasonable choice when the main goal is a compact comparison of centers and spreads, but it hides detailed shape. A histogram is reasonable when overlap and shape matter most, but the layers are busier. A violin plot combines a summary and smoothed shape, but its width may be unfamiliar and represents an estimate rather than raw counts.


### Investigate the longest taxi-in days

The figures show points or tails beyond the typical ranges. Now use `.nlargest()` to identify the days with the highest average taxi-in times. This lets us connect unusual marks in the figures back to actual observations.


In [ ]:
comparison.nlargest(8, "mean_taxi_in_min")[[
    "airport",
    "date",
    "mean_taxi_in_min"
]]


## Step 6 — Conclusion

Taxi-in distributions differed substantially across the four airports. The box plots place Miami and Orlando lowest, Dallas–Fort Worth higher, and Chicago O'Hare highest in both center and spread. Chicago's middle 50% of days lies largely above the middle ranges at the other airports.

The histogram reveals distinct concentrations, the box plot makes the increasing centers and spreads easy to compare, and the violin plot emphasizes Chicago's broader distribution. All four airports also had occasional days well above their typical taxi-in times.


## Step 7 — Limitation

- BTS data cover reported nonstop domestic flights, not every passenger or international flight.
- Airport differences can reflect layout, runway and gate assignments, traffic volume, weather, schedules, and the mix of airlines.
- Each day receives equal weight in the distribution even though daily flight counts differ.
- A daily average hides variation among flights within that day and gives each day equal weight.
- A longer taxi-in time does not, by itself, demonstrate inefficiency or identify its cause.


# A Second Analysis: Orlando Scheduled Arrivals by Season

Orlando is a major destination for vacations, conventions, and holiday travel. That motivates a follow-up question about when the airport handled the most scheduled arrivals. Because seasons contain slightly different numbers of days, we will compare the distribution of **daily scheduled arrivals** rather than seasonal totals.


## Step 1 — Question

> **During which season did Orlando International have the greatest typical number of scheduled arrivals per day in 2025?**

Travel demand may change with school calendars, holidays, weather, and tourism patterns. The analysis can identify the observed seasonal pattern, although it cannot determine why that pattern occurred.


## Discussion: predict the busiest arrival season

Which season do you predict will have the greatest typical number of scheduled arrivals per day at Orlando? What informed your prediction?


### Discussion notes

Predictions may reasonably refer to school breaks, holidays, weather, tourism patterns, or personal travel experience. The prediction is a hypothesis to compare with the data, not evidence by itself.


## Step 2 — Data

Select Orlando from the same daily airport file. Each row still represents one date at one airport.


In [ ]:
orlando = airport_days[
    airport_days["airport"] == "Orlando"
].copy()

orlando[["airport", "date", "scheduled_arrivals"]].head()


## Step 3 — Operation

### Create a season from a date

When the file was loaded, `parse_dates=["date"]` converted `date` from text into Pandas `datetime64` values. Datetime values have a `.dt` accessor that provides date-related properties and methods for an entire Series.

In `orlando["date"].dt.month`:

- `orlando["date"]` selects the date Series;
- `.dt` opens Pandas' datetime tools; and
- `.month` extracts the month number from every date, with January as 1 and December as 12.

We use meteorological seasons: winter is December–February, spring is March–May, summer is June–August, and fall is September–November.

A dictionary stores the season assigned to each month number. `.map()` looks up every extracted month number in that dictionary and returns its season. We save the result as a new `season` column.

Because this teaching file is already aggregated, we assign a season to each daily summary. With the original flight-level records, the same `.dt.month.map(season_by_month)` expression could assign a season to each flight before aggregation.


In [ ]:
season_by_month = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
}

orlando["season"] = (
    orlando["date"]
    .dt.month
    .map(season_by_month)
)

orlando[["date", "season"]].head()


Store the seasons in chronological order. Without an explicit order, text categories may follow their first appearance in the data instead of the intended seasonal cycle.

`.reindex(season_order)` rearranges the completed summary table to match this order. The same method puts the `value_counts()` result into chronological order. It changes how the result is displayed, not the underlying Orlando rows or scheduled-arrival values.


In [ ]:
season_order = ["Winter", "Spring", "Summer", "Fall"]

season_summary = (
    orlando.groupby("season")["scheduled_arrivals"]
    .describe()
    .reindex(season_order)
)

season_summary.round(1)


## Discussion: interpret the numerical summary

Based on the values calculated by `.describe()`, which season appears to have been busiest? What values support your answer?


### Discussion notes

Spring appears busiest by a typical day. It has the highest mean at about 465.7 scheduled arrivals and the highest median at about 468.5. The other seasons have lower means and medians.


## Step 4 — Check


In [ ]:
orlando[["date", "season", "scheduled_arrivals"]].isnull().sum()


In [ ]:
orlando["season"].value_counts().reindex(season_order)


The groups contain 90–92 days. These slight differences come from the number of calendar days in each season, not missing season assignments. All daily scheduled-arrival counts are positive. Comparing centers and complete distributions prevents a two-day difference in season length from deciding which season appears busiest.


## Step 5 — Evidence

### Histograms


In [ ]:
sns.histplot(
    data=orlando,
    x="scheduled_arrivals",
    hue="season",
    hue_order=season_order,
    element="step",
    multiple="layer",
    alpha=0.45,
    palette="deep"
)
plt.gca().get_legend().set_title("Season")
plt.title("Daily Scheduled Arrivals at Orlando by Season")
plt.xlabel("Scheduled arrivals per day")
plt.ylabel("Number of days")
plt.show()


## Discussion: compare the seasonal histograms

What can you conclude about seasonal scheduled arrivals from the histogram? What makes the comparison easy or difficult?


### Discussion notes

The histogram is difficult to compare because the four distributions overlap heavily. Spring appears shifted toward higher daily arrival counts and fall appears centered lowest, but the layered shapes make those differences hard to judge confidently.


### Box plots


In [ ]:
sns.boxplot(
    data=orlando,
    x="scheduled_arrivals",
    y="season",
    order=season_order,
    color="mediumseagreen"
)
plt.title("Daily Scheduled Arrivals at Orlando by Season")
plt.xlabel("Scheduled arrivals per day")
plt.ylabel("Season")
plt.show()


## Discussion: compare the seasonal box plots

What differences among the four seasons are easier to see in the box plots than in the histogram?


### Discussion notes

The box plots make the seasonal medians and middle 50% easier to compare. Spring has the highest median at about 468.5 scheduled arrivals per day, followed by winter at 444.5 and summer at 441.5. Fall has the lowest median at about 413 arrivals per day.


### Violin plots


In [ ]:
sns.violinplot(
    data=orlando,
    x="scheduled_arrivals",
    y="season",
    order=season_order,
    color="mediumpurple"
)
plt.title("Daily Scheduled Arrivals at Orlando by Season")
plt.xlabel("Scheduled arrivals per day")
plt.ylabel("Season")
plt.show()


## Discussion: compare the seasonal violin plots

What does the violin plot add to the seasonal comparison? What remains difficult to interpret?


### Discussion notes

The violin plots combine a smoothed view of shape with an interior box-plot summary. They preserve more information about concentrations than the box plots, but the overlapping seasonal centers and the meaning of violin width may still be difficult for an unfamiliar audience to interpret. The smooth outline is an estimate and may extend beyond observed values.


## Discussion: choose the clearest figure

Which figure does the clearest job of comparing the number of scheduled arrivals by season? Explain your choice.


### Discussion notes

The box plot is likely the clearest figure for this question because the seasonal medians and middle ranges can be compared directly without overlapping layers. A student may reasonably choose another figure if the explanation accurately identifies what it communicates well and what it makes harder to see.


## Step 6 — Conclusion

Spring was Orlando's busiest season by the typical number of scheduled arrivals per day in 2025. The spring median was approximately 468.5 scheduled arrivals per day, compared with 444.5 in winter, 441.5 in summer, and 413.0 in fall. Spring also had the highest mean at about 465.7 scheduled arrivals per day. The distributions overlap, so not every spring day was busier than every day in another season.


## Step 7 — Limitation

- The analysis describes only 2025, so it cannot establish a recurring seasonal pattern.
- Meteorological seasons are course-defined categories; another definition could change membership near the boundaries.
- Season is associated with many changing conditions, including holidays, school calendars, weather, airline schedules, and tourism demand.
- Scheduled arrivals describe planned flight volume, not passenger counts, completed flights, or airport capacity use.
- Each row summarizes one date, so the analysis cannot explain differences among individual flights.
- The comparison does not establish that spring itself caused higher scheduled traffic.


## Discussion: compare the seasonal distributions

1. The seasons contain between 90 and 92 days. Why could comparing seasonal totals make a longer season appear busier simply because it contains more days?

2. Does “spring was busiest” mean that every spring day had more scheduled arrivals than every day in another season?

3. What does the scheduled-arrival feature measure, and what does it leave out?

4. Why is one year insufficient to claim that spring is always Orlando's busiest season?


### Discussion notes

1. A longer season has more opportunities to accumulate scheduled arrivals. Comparing daily distributions separates the typical number of arrivals from the number of days included in the group.

2. No. The distributions overlap. The statement compares seasonal centers, not every pair of dates.

3. It measures the number of flights scheduled to arrive at Orlando on each date. It does not report passengers, completed arrivals, or how full the flights were.

4. A single year may reflect unusual schedules, weather, events, or travel behavior. Multiple years are needed to evaluate whether the pattern recurs.


## Key takeaways

- Define precisely what one observation and one quantitative value represent.
- Confirm that values, units, and exclusions match the research question.
- Histograms, box plots, and violin plots emphasize different distribution features.
- Compare center, spread, shape, overlap, and unusual values—not only averages.
- Conclusions should describe association, not unsupported causation.
- Pandas datetime tools can turn dates into analytically useful calendar categories.
- When groups contain different numbers of days, compare daily distributions rather than letting unequal group sizes determine a total.
